# Verify paper numbers — NeurIPS 2026 paper #1296

**"When Classical Inference Fails: A Multi-Criteria Diagnostic for Causal Discovery on Latent-Factor Panels"**

This notebook asserts every numerical claim in the paper against the canonical CSV / JSON outputs in the reproducibility tarball. **No re-computation is performed.**

For each claim, the notebook does two extractions at runtime:
1. **Paper extraction** — pulls the value from the paper's `.tex` source files (e.g. a cell of Table 2, an inline value like `sup-Wald = 10.8`).
2. **Canonical extraction** — pulls the corresponding value from the depository's CSV/JSON outputs.

Then it compares the two within tolerance. **Neither side hard-codes expected values.** The mapping between paper claims and canonical sources is defined by paired extractor functions in `verifier_core.py` — open that file to see exactly what each check is doing.

## How to use this notebook

**Option A — On Colab (point-and-click).** Run the cells in order. Cell 2 will prompt you to upload these four files via the file picker:
- `neurips_1296_bundle.tar.gz` (the tarball — contains both the depository and the paper sources)
- `paper_extractors.py`
- `canonical_extractors.py`
- `verifier_core.py`

**Option B — Locally (e.g., with `jupyter notebook`).** Edit the path variables in cell 2 to point to the files on your machine, then run cells 3 onward.

**Option C — From terminal.** Use `python verify_paper_numbers.py --tarball /path/to/bundle.tar.gz` instead. (This notebook is functionally equivalent to that script.)

## Tolerance defaults

| Tolerance | Used for                                                  |
|----------:|:----------------------------------------------------------|
|   ±0.001  | individual values where paper is precise (3+ decimals)    |
|   ±0.005  | cross-method aggregates                                   |
|   ±0.01   | "≈ X" claims rounded to 2 decimal places                  |
|   ±0.05   | "≈ X" claims rounded to 1 decimal place                   |
|  `≤` only | p-values stated as `p ≤ X` (checked as inequality)        |


## 1. Detect environment (Colab vs. local)


In [ ]:
import os, sys

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

print('Colab detected.' if ON_COLAB else 'Local Jupyter detected.')

## 2. Locate inputs (tarball + verifier modules)

On Colab: file picker pops up. Upload all four files when prompted.
Locally: edit the paths in the `else` branch.


In [ ]:
REQUIRED = {
    'neurips_1296_bundle.tar.gz',
    'paper_extractors.py',
    'canonical_extractors.py',
    'verifier_core.py',
}

if ON_COLAB:
    from google.colab import files
    print('Please upload:')
    for f in sorted(REQUIRED):
        print(f'  - {f}')
    print()
    uploaded = files.upload()
    missing = REQUIRED - set(uploaded.keys())
    if missing:
        raise RuntimeError(f'Missing uploads: {sorted(missing)}. Re-run this cell to retry.')
    TARBALL_PATH        = '/content/neurips_1296_bundle.tar.gz'
    PAPER_EXTRACTORS    = '/content/paper_extractors.py'
    CANONICAL_EXTRACT   = '/content/canonical_extractors.py'
    VERIFIER_CORE       = '/content/verifier_core.py'
else:
    # Local: edit these to point at the files on your filesystem.
    TARBALL_PATH        = './neurips_1296_bundle.tar.gz'
    PAPER_EXTRACTORS    = './paper_extractors.py'
    CANONICAL_EXTRACT   = './canonical_extractors.py'
    VERIFIER_CORE       = './verifier_core.py'

for path in (TARBALL_PATH, PAPER_EXTRACTORS, CANONICAL_EXTRACT, VERIFIER_CORE):
    if not os.path.exists(path):
        raise FileNotFoundError(f'Not found: {path}')
    print(f'  {path}  ({os.path.getsize(path):,} bytes)')


## 3. Install dependencies and import verifier modules

We need `pandas` and `numpy`. Both are usually pre-installed on Colab.


In [ ]:
import subprocess, importlib
for mod in ['pandas', 'numpy']:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', mod])

# Make the three verifier modules importable
import shutil
for src in (PAPER_EXTRACTORS, CANONICAL_EXTRACT, VERIFIER_CORE):
    name = os.path.basename(src)
    if os.path.abspath(src) != os.path.abspath('./' + name):
        shutil.copy(src, './' + name)

if '.' not in sys.path:
    sys.path.insert(0, '.')

import paper_extractors as pe
import verifier_core as vc

print(f'Loaded verifier_core with {len(vc.CHECKS)} checks.')

## 4. Extract the tarball

Locates the depository root and the paper sources. The new tarball layout is:

```
NeurIPS2026_1296/
├── README.md
├── data/, src/, analysis/, *_outputs/, ...    (canonical depository)
└── paper/                                      (paper sources, .tex files)
    ├── main.tex
    ├── appendix_A.tex, appendix_B.tex, ...
    ├── neurips_2026.sty, references.bib
    └── main.pdf
```


In [ ]:
import tarfile, tempfile
from pathlib import Path

EXTRACT_DIR = Path(tempfile.mkdtemp(prefix='verify_1296_'))
print(f'Extracting tarball to: {EXTRACT_DIR}')

with tarfile.open(TARBALL_PATH, 'r:*') as tf:
    try:
        tf.extractall(EXTRACT_DIR, filter='data')
    except TypeError:
        tf.extractall(EXTRACT_DIR)

# Find depository root (directory containing multi_criteria_outputs/)
def find_depository_root(d):
    sentinel = 'multi_criteria_outputs'
    if (d / sentinel).is_dir():
        return d
    for child in d.iterdir():
        if child.is_dir() and (child / sentinel).is_dir():
            return child
    raise FileNotFoundError(f'No depository root in {d}')

DEPOSITORY = find_depository_root(EXTRACT_DIR)
PAPER_DIR = DEPOSITORY / 'paper'
if not PAPER_DIR.is_dir() or not any(PAPER_DIR.glob('*.tex')):
    raise FileNotFoundError(f'No paper sources at {PAPER_DIR}')

print(f'  Depository:  {DEPOSITORY}')
print(f'  Paper dir:   {PAPER_DIR}')

## 5. Load paper sources and run all checks


In [ ]:
paper_sources = pe.load_paper_sources(PAPER_DIR)
print(f'Loaded paper files: {sorted(paper_sources.keys())}\n')

results = vc.run_all(paper_sources, DEPOSITORY)
summary = vc.summarize(results)

n_pass = summary['pass']
n_fail = summary['fail']
n_err  = summary['error']
n_total = summary['total']

print('=' * 66)
print('  SUMMARY')
print('=' * 66)
print(f'  Total:   {n_total}')
print(f'  Passed:  {n_pass}')
print(f'  Failed:  {n_fail}')
print(f'  Errors:  {n_err}')
print('=' * 66)

## 6. Show FAIL / ERROR details (if any)


In [ ]:
if n_fail > 0 or n_err > 0:
    for r in results:
        if r.status != 'PASS':
            print(vc.format_result(r))
            print()
else:
    print('All checks pass — no FAILs or ERRORs to show.')

## 7. (Optional) Show every PASS line


In [ ]:
# Uncomment to list all passing checks:
# for r in results:
#     if r.status == 'PASS':
#         print(vc.format_result(r))

## 8. (Optional) Inspect the source map for any check

Each check has `paper_loc` (where in the paper the claim appears) and a `description`. The `paper` and `canonical` callables tell you exactly what is being extracted from each side.


In [ ]:
# Example: the first 5 checks and their paper locations
for c in vc.CHECKS[:5]:
    print(f'  • {c.description}')
    print(f'    paper_loc: {c.paper_loc}')
    print(f'    tol:       {c.tol}, comparator: {c.comparator}')
    print()

## Cleanup


In [ ]:
# Uncomment to delete the temp extraction directory:
# import shutil
# shutil.rmtree(EXTRACT_DIR)
# print(f'Deleted {EXTRACT_DIR}')